# Layout Experiments: Alternative Visualization Arrangements

This notebook explores different layout options for the comprehensive plots:

## Layouts to Test

### 1. **Standard Improved** (Remove Risk_After, consolidate row 2)
- Row 1: Validity_%, Solved_%, Time, [empty or future metric]
- Row 2: Risk_Before, Risk_Reduction_%, Avg_NChanged_All, Avg_NChanged_Valid
- Row 3: Avg_Gower_All, Low_Gower_%_All, High_Gower_%_All, Median_Gower_All
- Row 4: Avg_Gower_Valid, Low_Gower_%_Valid, High_Gower_%_Valid, Min_Gower_Valid

### 2. **Success Pipeline** (Story-focused)
- Row 1 (Funnel): Validity_%, Solved_%, Risk_Reduction_%, Actionable_%
- Row 2 (Effort): Time, Avg_NChanged_All, Avg_NChanged_Valid, Risk_Before
- Row 3 (Quality - Distance): Avg_Gower_All, Median_Gower_All, Avg_Gower_Valid, Min_Gower_Valid
- Row 4 (Quality - Bounds): Low_Gower_%_All, High_Gower_%_All, Low_Gower_%_Valid, High_Gower_%_Valid

### 3. **All vs Valid Comparison** (Systematic)
- Row 1 (All CFs - Performance): Validity_%, Time, Avg_NChanged_All, Avg_Gower_All
- Row 2 (All CFs - Distribution): Low_Gower_%_All, High_Gower_%_All, Median_Gower_All, [Risk_Before]
- Row 3 (Valid CFs - Performance): Solved_%, Risk_Reduction_%, Avg_NChanged_Valid, Avg_Gower_Valid
- Row 4 (Valid CFs - Distribution): Low_Gower_%_Valid, High_Gower_%_Valid, Min_Gower_Valid, Actionable_%

### 4. **Efficiency Focus** (Production-oriented)
- Row 1 (Conversion): Validity_%, Solved_%, Actionable_%, Risk_Reduction_%
- Row 2 (Cost): Time, Avg_NChanged_All, Avg_NChanged_Valid, Risk_Before
- Row 3 (Proximity): Avg_Gower_All, Avg_Gower_Valid, Median_Gower_All, Min_Gower_Valid
- Row 4 (Extremes): Low_Gower_%_All, High_Gower_%_All, Low_Gower_%_Valid, High_Gower_%_Valid

---

In [ ]:
# Import libraries
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

ROOT_DIR = Path(os.getenv("PROJECT_ROOT"))
RESULTS_CSV_PATH = ROOT_DIR / "analysis" / "gen_2_summary.csv"

print("root exists: ", ROOT_DIR.exists())
print("results is file: ", RESULTS_CSV_PATH.is_file())

In [ ]:
# Add project root to path for imports
import sys
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

In [ ]:
df = pd.read_csv(RESULTS_CSV_PATH)
df.head(3)

## Prepare Data

We'll prepare both "good" and "bad" experiment datasets for testing different layouts.

In [ ]:
# Filter for good experiments (Baseline + Optimized)
df_good = df[~df['csv_path'].str.contains('SMOTE|weighted', regex=True)].copy()

# Create category
df_good['category'] = df_good['csv_path'].apply(
    lambda x: 'Baseline' if 'baseline' in x else 'Optimized XGBoost'
)

print(f"Good experiments: {len(df_good)}")

In [ ]:
# Filter for bad experiments (SMOTE + Weighted)
df_bad = df[df['csv_path'].str.contains('SMOTE|weighted', regex=True)].copy()

# Create category
df_bad['category'] = df_bad['csv_path'].apply(
    lambda x: 'SMOTE (Base)' if 'SMOTE/base_predictors' in x
              else 'SMOTE (Gridsearched)' if 'SMOTE/gridsearched' in x
              else 'Weighted'
)

print(f"Bad experiments: {len(df_bad)}")

## Helper Function: Prepare Data for Custom Layouts

This function allows us to specify which metrics go in which row.

In [ ]:
def prepare_data_custom_layout(df, row_metrics_dict):
    """
    Prepare data with custom layout.

    Parameters:
    -----------
    df : DataFrame
        Input dataframe with prepared columns
    row_metrics_dict : dict
        Dictionary mapping row names to list of metrics
        e.g., {'row1': ['Validity_%', 'Solved_%', 'Time', 'Avg_NChanged_All']}

    Returns:
    --------
    dict : Dictionary of melted dataframes for each row
    """
    result = {}

    for row_name, metrics in row_metrics_dict.items():
        # Filter out None values (empty slots)
        metrics = [m for m in metrics if m is not None]

        df_melted = df.melt(
            id_vars=["Category", "Model", "Threshold"],
            value_vars=metrics,
            var_name="Metric",
            value_name="Value"
        )
        result[row_name] = df_melted

    return result

In [ ]:
def prepare_common_columns(df):
    """Prepare all common columns needed for visualization."""
    df = df.copy()

    # Basic metrics
    df["Validity_%"] = df["validity_%"].str.rstrip('%').astype(float)
    df["Solved_%"] = df["solved_%"].str.rstrip('%').astype(float)
    df["Actionable_%"] = df["actionable_%"].str.rstrip('%').astype(float)
    df["Time"] = df["total_gen_time_sec"]

    # Labels
    df["Category"] = df["category"]
    df["Model"] = df["ml_model_type"]
    df["Threshold"] = df["stopping_threshold"]

    # Number of changes
    df["Avg_NChanged_All"] = df["avg_nchanged_all"]
    df["Avg_NChanged_Valid"] = df["avg_nchanged_valid"]

    # Gower distance - ALL
    df["Avg_Gower_All"] = df["avg_gower_all"]
    low_col_all = [c for c in df.columns if c.startswith("low_gower_<") and c.endswith("_%_all")][0]
    high_col_all = [c for c in df.columns if c.startswith("high_gower_>") and c.endswith("_%_all")][0]
    df["Low_Gower_%_All"] = df[low_col_all].str.rstrip('%').astype(float)
    df["High_Gower_%_All"] = df[high_col_all].str.rstrip('%').astype(float)
    df["Median_Gower_All"] = df["gower_median_all"]

    # Gower distance - VALID
    df["Avg_Gower_Valid"] = df["avg_gower_valid"]
    low_col_valid = [c for c in df.columns if c.startswith("low_gower_<") and c.endswith("_%_valid")][0]
    high_col_valid = [c for c in df.columns if c.startswith("high_gower_>") and c.endswith("_%_valid")][0]
    df["Low_Gower_%_Valid"] = df[low_col_valid].str.rstrip('%').astype(float)
    df["High_Gower_%_Valid"] = df[high_col_valid].str.rstrip('%').astype(float)
    df["Min_Gower_Valid"] = df["min_gower_valid"]

    # Risk metrics
    df["Risk_Before"] = df["avg_risk_before"] * 100
    df["Risk_After"] = df["avg_risk_after"] * 100
    df["Risk_Reduction_%"] = df["risk_reduction_%"]

    return df

In [ ]:
# Prepare both datasets
df_good_prep = prepare_common_columns(df_good)
df_bad_prep = prepare_common_columns(df_bad)

print("Data prepared successfully")
print(f"Good: {len(df_good_prep)} rows")
print(f"Bad: {len(df_bad_prep)} rows")

## Layout 1: Standard Improved (Risk_After removed)

This is the proposed modification to the current layout:
- Remove Risk_After (redundant with Risk_Before + Risk_Reduction_%)
- Move both Avg_NChanged metrics to Row 2
- Creates thematic "Risk & Effort" row

In [ ]:
# Define Layout 1
layout_1 = {
    'row1': ['Validity_%', 'Solved_%', 'Time', None],  # None for empty slot
    'row2': ['Risk_Before', 'Risk_Reduction_%', 'Avg_NChanged_All', 'Avg_NChanged_Valid'],
    'row3': ['Avg_Gower_All', 'Low_Gower_%_All', 'High_Gower_%_All', 'Median_Gower_All'],
    'row4': ['Avg_Gower_Valid', 'Low_Gower_%_Valid', 'High_Gower_%_Valid', 'Min_Gower_Valid']
}

# Test with good data
data_layout_1_good = prepare_data_custom_layout(df_good_prep, layout_1)

print("Layout 1 - Good experiments:")
for row_name, df_row in data_layout_1_good.items():
    print(f"  {row_name}: {df_row.shape}")

**Note:** To actually visualize Layout 1, you would need to create a modified plotting function that handles the empty slot in row 1. For now, we're just confirming the data structure works.

## Layout 2: Success Pipeline

Story-focused layout that follows the counterfactual generation journey:
1. Funnel metrics (what succeeded?)
2. Effort metrics (what did it cost?)
3. Quality metrics - distance measures
4. Quality metrics - distribution extremes

In [ ]:
# Define Layout 2
layout_2 = {
    'row1_funnel': ['Validity_%', 'Solved_%', 'Risk_Reduction_%', 'Actionable_%'],
    'row2_effort': ['Time', 'Avg_NChanged_All', 'Avg_NChanged_Valid', 'Risk_Before'],
    'row3_quality_dist': ['Avg_Gower_All', 'Median_Gower_All', 'Avg_Gower_Valid', 'Min_Gower_Valid'],
    'row4_quality_bounds': ['Low_Gower_%_All', 'High_Gower_%_All', 'Low_Gower_%_Valid', 'High_Gower_%_Valid']
}

# Test with good data
data_layout_2_good = prepare_data_custom_layout(df_good_prep, layout_2)

print("Layout 2 - Success Pipeline (Good experiments):")
for row_name, df_row in data_layout_2_good.items():
    print(f"  {row_name}: {df_row.shape}")

## Layout 3: All vs Valid Comparison

Systematic comparison between "All CFs" and "Valid CFs only":
- Rows 1-2: All generated counterfactuals
- Rows 3-4: Valid counterfactuals only

In [ ]:
# Define Layout 3
layout_3 = {
    'row1_all_perf': ['Validity_%', 'Time', 'Avg_NChanged_All', 'Avg_Gower_All'],
    'row2_all_dist': ['Low_Gower_%_All', 'High_Gower_%_All', 'Median_Gower_All', 'Risk_Before'],
    'row3_valid_perf': ['Solved_%', 'Risk_Reduction_%', 'Avg_NChanged_Valid', 'Avg_Gower_Valid'],
    'row4_valid_dist': ['Low_Gower_%_Valid', 'High_Gower_%_Valid', 'Min_Gower_Valid', 'Actionable_%']
}

# Test with good data
data_layout_3_good = prepare_data_custom_layout(df_good_prep, layout_3)

print("Layout 3 - All vs Valid (Good experiments):")
for row_name, df_row in data_layout_3_good.items():
    print(f"  {row_name}: {df_row.shape}")

## Layout 4: Efficiency Focus

Production-oriented layout emphasizing practical concerns:
1. Conversion rates (what works?)
2. Cost metrics (what resources needed?)
3. Proximity measures (how close are CFs?)
4. Extreme cases (edge case analysis)

In [ ]:
# Define Layout 4
layout_4 = {
    'row1_conversion': ['Validity_%', 'Solved_%', 'Actionable_%', 'Risk_Reduction_%'],
    'row2_cost': ['Time', 'Avg_NChanged_All', 'Avg_NChanged_Valid', 'Risk_Before'],
    'row3_proximity': ['Avg_Gower_All', 'Avg_Gower_Valid', 'Median_Gower_All', 'Min_Gower_Valid'],
    'row4_extremes': ['Low_Gower_%_All', 'High_Gower_%_All', 'Low_Gower_%_Valid', 'High_Gower_%_Valid']
}

# Test with good data
data_layout_4_good = prepare_data_custom_layout(df_good_prep, layout_4)

print("Layout 4 - Efficiency Focus (Good experiments):")
for row_name, df_row in data_layout_4_good.items():
    print(f"  {row_name}: {df_row.shape}")

## Test with Bad Experiments

Let's verify Layout 2 (Success Pipeline) with the SMOTE/Weighted data as well.

In [ ]:
# Test Layout 2 with bad experiments
data_layout_2_bad = prepare_data_custom_layout(df_bad_prep, layout_2)

print("Layout 2 - Success Pipeline (Bad experiments):")
for row_name, df_row in data_layout_2_bad.items():
    print(f"  {row_name}: {df_row.shape}")

## Summary & Recommendations

### Layout Characteristics:

**Layout 1 (Standard Improved):**
- ✅ Minimal change from current
- ✅ Removes redundant Risk_After
- ✅ Consolidates "Risk & Effort" story in row 2
- ⚠️ Empty slot in row 1 (could add future metric)

**Layout 2 (Success Pipeline):**
- ✅ Best for presentations/storytelling
- ✅ Logical flow: Success → Effort → Quality
- ✅ All 16 slots filled
- ❌ Less obvious "All vs Valid" distinction

**Layout 3 (All vs Valid):**
- ✅ Systematic comparison structure
- ✅ Best for technical analysis
- ✅ Clear separation of perspectives
- ❌ May be less intuitive for general audience

**Layout 4 (Efficiency Focus):**
- ✅ Production/business oriented
- ✅ Emphasizes practical concerns
- ✅ Good for stakeholder presentations
- ❌ Less emphasis on quality details

### My Recommendation:

1. **Implement Layout 1 first** - cleanest improvement to current design
2. **Try Layout 2 for paper/presentation** - tells the complete story
3. **Keep current layout available** - some may prefer seeing Risk_After explicitly

### Next Steps:

To actually implement these layouts, you would need to:
1. Create modified plotting functions that handle these metric arrangements
2. Update label dictionaries to match new row themes
3. Adjust scale configurations per row if needed
4. Test visual clarity with actual plots

Would you like me to implement plotting functions for any of these layouts?